# Telecom X – Parte 2: Predicción de Cancelación (Churn)

Modelos de predicción sobre churn de clientes en TelecomX.

## 📚 Diccionario de datos del dataset

- customerID: número de identificación único de cada cliente
- Churn: si el cliente dejó o no la empresa
- gender: género (masculino y femenino)
- SeniorCitizen: información sobre si un cliente tiene o no una edad igual o mayor a 65 años
- Partner: si el cliente tiene o no una pareja
- Dependents: si el cliente tiene o no dependientes
- tenure: meses de contrato del cliente
- PhoneService: suscripción al servicio telefónico
- MultipleLines: suscripción a más de una línea telefónica
- InternetService: suscripción a un proveedor de internet
- OnlineSecurity: suscripción adicional de seguridad en línea
- OnlineBackup: suscripción adicional de respaldo en línea
- DeviceProtection: suscripción adicional de protección del dispositivo
- TechSupport: suscripción adicional de soporte técnico, menor tiempo de espera
- StreamingTV: suscripción de televisión por cable
- StreamingMovies: suscripción de streaming de películas
- Contract: tipo de contrato
- PaperlessBilling: si el cliente prefiere recibir la factura en línea
- PaymentMethod: forma de pago
- Charges.Monthly: total de todos los servicios del cliente por mes
- Charges.Total: total gastado por el cliente

## 📌 Extracción

***Nota***: Vamos a volver a cargar los datos originales, ya que las transformaciones que realizamos en la parte 1 del proyecto no son adecuadas para los modelos de machine-learning que vamos a realizar en este proyecto.

In [57]:
# Importamos pandas, abrimos el json desde url y lo cargamos en un df

import pandas as pd

url = 'https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/refs/heads/main/TelecomX_Data.json'

datos_raw = pd.read_json(url)
datos_raw.head()

,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
2,0004-TLHLJ,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
3,0011-IGKFF,Yes,"{'gender': 'Male', 'SeniorCitizen': 1, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
4,0013-EXCHZ,Yes,"{'gender': 'Female', 'SeniorCitizen': 1, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


- Detectamos que hay columnas anidadas. Vamos a importar los datos con ***requests*** para manejar errores de link y luego normalizar el json con ***json_normalize*** antes de cargarlo a un df.

In [58]:
import pandas as pd
import requests
import json

# 1. Definimos la URL del json
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/refs/heads/main/TelecomX_Data.json"

try:
    # 2. Hacemos la petición (request) para obtener el texto crudo
    print("⏳ Descargando datos...")
    response = requests.get(url)
    response.raise_for_status() # Esto nos avisa si el link está roto (error 404, etc.)

    # 3. Convertimos ese texto a un objeto de Python
    data = response.json()
    print("✅ Datos descargados y leídos correctamente.")

    # 4. Normalizamos (aplanamos) el JSON. json_normalize convierte estructuras anidadas en columnas separadas.
    df = pd.json_normalize(data)

    # 5. Mostramos las primeras filas y las columnas resultantes
    print(f"\nEl DataFrame tiene {df.shape[0]} filas y {df.shape[1]} columnas.")
    print("\n--- Primeras 5 filas ---")
    print(df.head())

except requests.exceptions.HTTPError as e:
    print(f"❌ Error al descargar el archivo: {e}")
except json.JSONDecodeError:
    print(f"❌ El contenido descargado no es un JSON válido.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado: {e}")

⏳ Descargando datos...
✅ Datos descargados y leídos correctamente.

El DataFrame tiene 7267 filas y 21 columnas.

--- Primeras 5 filas ---
   customerID Churn customer.gender  customer.SeniorCitizen customer.Partner  \
0  0002-ORFBO    No          Female                       0              Yes   
1  0003-MKNFE    No            Male                       0               No   
2  0004-TLHLJ   Yes            Male                       0               No   
3  0011-IGKFF   Yes            Male                       1              Yes   
4  0013-EXCHZ   Yes          Female                       1              Yes   

  customer.Dependents  customer.tenure phone.PhoneService phone.MultipleLines  \
0                 Yes                9                Yes                  No   
1                  No                9                Yes                 Yes   
2                  No                4                Yes                  No   
3                  No               13                Ye

## 🔧 Transformación

### Verificación de inconsistencias

In [59]:
# Priumer analisis exploratorio
df.tail(5)

,customerID,Churn,customer.gender,customer.SeniorCitizen,customer.Partner,customer.Dependents,customer.tenure,phone.PhoneService,phone.MultipleLines,internet.InternetService,...,internet.OnlineBackup,internet.DeviceProtection,internet.TechSupport,internet.StreamingTV,internet.StreamingMovies,account.Contract,account.PaperlessBilling,account.PaymentMethod,account.Charges.Monthly,account.Charges.Total
7262,9987-LUTYD,No,Female,0,No,No,13,Yes,No,DSL,...,No,No,Yes,No,No,One year,No,Mailed check,55.15,742.9
7263,9992-RRAMN,Yes,Male,0,Yes,No,22,Yes,Yes,Fiber optic,...,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,85.10,1873.7
7264,9992-UJOEL,No,Male,0,No,No,2,Yes,No,DSL,...,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,50.30,92.75
7265,9993-LHIEB,No,Male,0,Yes,Yes,67,Yes,No,DSL,...,No,Yes,Yes,No,Yes,Two year,No,Mailed check,67.85,4627.65
7266,9995-HOTOH,No,Male,0,Yes,Yes,63,No,No phone service,DSL,...,Yes,Yes,No,Yes,Yes,Two year,No,Electronic check,59.00,3707.6


- Valores repetidos: vamos a verificar si hayt registros duplicados en el df.

In [60]:
print(f'Registros duplicados: {df.duplicated().sum()}.')

Registros duplicados: 0.


- Valors nulos: verificamos si hay valores nulos.

In [61]:
print(f'Registros núlos: {df.isnull().sum()}.')

Registros núlos: customerID                   0
Churn                        0
customer.gender              0
customer.SeniorCitizen       0
customer.Partner             0
customer.Dependents          0
customer.tenure              0
phone.PhoneService           0
phone.MultipleLines          0
internet.InternetService     0
internet.OnlineSecurity      0
internet.OnlineBackup        0
internet.DeviceProtection    0
internet.TechSupport         0
internet.StreamingTV         0
internet.StreamingMovies     0
account.Contract             0
account.PaperlessBilling     0
account.PaymentMethod        0
account.Charges.Monthly      0
account.Charges.Total        0
dtype: int64.


- Valores vacios o en blanco.

In [62]:
df.apply(lambda x: x.astype(str).str.strip() == '').sum()

,0
customerID,0
Churn,224
customer.gender,0
customer.SeniorCitizen,0
customer.Partner,0
customer.Dependents,0
customer.tenure,0
phone.PhoneService,0
phone.MultipleLines,0
internet.InternetService,0


### Manejo de inconsistencias

In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customerID                 7267 non-null   object 
 1   Churn                      7267 non-null   object 
 2   customer.gender            7267 non-null   object 
 3   customer.SeniorCitizen     7267 non-null   int64  
 4   customer.Partner           7267 non-null   object 
 5   customer.Dependents        7267 non-null   object 
 6   customer.tenure            7267 non-null   int64  
 7   phone.PhoneService         7267 non-null   object 
 8   phone.MultipleLines        7267 non-null   object 
 9   internet.InternetService   7267 non-null   object 
 10  internet.OnlineSecurity    7267 non-null   object 
 11  internet.OnlineBackup      7267 non-null   object 
 12  internet.DeviceProtection  7267 non-null   object 
 13  internet.TechSupport       7267 non-null   objec

- Lo primero que detectamos es que la columna account.Charges.Total esta como string en vez de float64. Vamos a arreglarlo.


In [64]:
# Convertimos la columna forzando los errores a NaN
df['account.Charges.Total'] = pd.to_numeric(df['account.Charges.Total'], errors='coerce')

# Verificamos cuántos valores se convirtieron en NaN
nulos = df['account.Charges.Total'].isna().sum()
print(f"Se encontraron {nulos} valores que no eran números y se convirtieron a NaN.")

Se encontraron 11 valores que no eran números y se convirtieron a NaN.


In [65]:
# Como son pocos los valores nulos, vamos a deshecharlos para no alterar el linaje de datos
df = df.dropna()

In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7256 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customerID                 7256 non-null   object 
 1   Churn                      7256 non-null   object 
 2   customer.gender            7256 non-null   object 
 3   customer.SeniorCitizen     7256 non-null   int64  
 4   customer.Partner           7256 non-null   object 
 5   customer.Dependents        7256 non-null   object 
 6   customer.tenure            7256 non-null   int64  
 7   phone.PhoneService         7256 non-null   object 
 8   phone.MultipleLines        7256 non-null   object 
 9   internet.InternetService   7256 non-null   object 
 10  internet.OnlineSecurity    7256 non-null   object 
 11  internet.OnlineBackup      7256 non-null   object 
 12  internet.DeviceProtection  7256 non-null   object 
 13  internet.TechSupport       7256 non-null   object 
 1

- Ahora solo nos queda la columna ***Churn*** con valores vacios. Vamos a eliminarlos.

In [67]:
df = df[df['Churn'].str.strip() != '']
print('Números de filas luego de eliminar vacios en churn: ', len(df))

Números de filas luego de eliminar vacios en churn:  7032


In [68]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customerID                 7032 non-null   object 
 1   Churn                      7032 non-null   object 
 2   customer.gender            7032 non-null   object 
 3   customer.SeniorCitizen     7032 non-null   int64  
 4   customer.Partner           7032 non-null   object 
 5   customer.Dependents        7032 non-null   object 
 6   customer.tenure            7032 non-null   int64  
 7   phone.PhoneService         7032 non-null   object 
 8   phone.MultipleLines        7032 non-null   object 
 9   internet.InternetService   7032 non-null   object 
 10  internet.OnlineSecurity    7032 non-null   object 
 11  internet.OnlineBackup      7032 non-null   object 
 12  internet.DeviceProtection  7032 non-null   object 
 13  internet.TechSupport       7032 non-null   object 
 1

### Estandarización y transformación de datos

- Cambiamos el nombre de las columnas para facilitar la lectura, especialmente de los stakeholders no técnicos.

In [69]:
columnas = {
    'customerID': 'id',
    'Churn': 'Churn',
    'customer.gender': 'Gender',
    'customer.SeniorCitizen': 'SeniorCitizen',
    'customer.Partner': 'Partner',
    'customer.Dependents': 'Dependents',
    'customer.tenure': 'Tenure',
    'phone.PhoneService': 'PhoneService',
    'phone.MultipleLines': 'MultipleLines',
    'internet.InternetService': 'InternetService',
    'internet.OnlineSecurity': 'OnlineSecurity',
    'internet.OnlineBackup': 'OnlineBackup',
    'internet.DeviceProtection': 'DeviceProtection',
    'internet.TechSupport': 'TechSupport',
    'internet.StreamingTV': 'StreamingTV',
    'internet.StreamingMovies': 'StreamingMovies',
    'account.Contract': 'Contract',
    'account.PaperlessBilling': 'PaperlessBilling',
    'account.PaymentMethod': 'PaymentMethod',
    'account.Charges.Monthly': 'Charges.Monthly',
    'account.Charges.Total': 'Charges.Total',
    }

df.rename(columns=columnas, inplace=True)
df.head()

,id,Churn,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,...,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.30
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.40
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.0,1237.85
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.9,267.40


- Vamos a inspeccionar los valores unicos de cada columna, para detectar posibles desvios o errores de carga. Vamos a seleccionar las columnas categoricas y explorarlas una a una.

In [70]:
columnas_categoricas = list(df.columns[1:19]) # No incluimos el ID, porque son todos valores unicos...

columnas_categoricas

['Churn',
 'Gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'Tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

In [71]:
# Sacamos tiempo_contrato (tenure) porque son valores diferentes..
columnas_categoricas.remove('Tenure')
columnas_categoricas

['Churn',
 'Gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

In [72]:
# Verificación de valores unicos de cada columna categorica

for col in columnas_categoricas:
    print(f"Valores únicos en la columna {col}: {df[col].unique()}.")

Valores únicos en la columna Churn: ['No' 'Yes'].
Valores únicos en la columna Gender: ['Female' 'Male'].
Valores únicos en la columna SeniorCitizen: [0 1].
Valores únicos en la columna Partner: ['Yes' 'No'].
Valores únicos en la columna Dependents: ['Yes' 'No'].
Valores únicos en la columna PhoneService: ['Yes' 'No'].
Valores únicos en la columna MultipleLines: ['No' 'Yes' 'No phone service'].
Valores únicos en la columna InternetService: ['DSL' 'Fiber optic' 'No'].
Valores únicos en la columna OnlineSecurity: ['No' 'Yes' 'No internet service'].
Valores únicos en la columna OnlineBackup: ['Yes' 'No' 'No internet service'].
Valores únicos en la columna DeviceProtection: ['No' 'Yes' 'No internet service'].
Valores únicos en la columna TechSupport: ['Yes' 'No' 'No internet service'].
Valores únicos en la columna StreamingTV: ['Yes' 'No' 'No internet service'].
Valores únicos en la columna StreamingMovies: ['No' 'Yes' 'No internet service'].
Valores únicos en la columna Contract: ['One ye

- No se observan grandes irregularidades. Pero vamos a unificar el formato de las columnas. Primero nos encargamos de las columnas que solo tienen Yes No.

In [73]:
# 1. Definimos el mapeo  para Yes / No - 1 / 0
mapa_booleano = {'Yes': 1, 'No': 0}

# 2. Identificamos las columnas que contienen 'Yes' y 'No'
#    (excluimos las que tienen 3 opciones como "No internet service" y )

cols_yes_no = []
for col in df.columns:
    # Revisamos si es tipo objeto (texto/string) y si sus valores únicos son solo Yes/No o nulo
    if df[col].dtype == 'object':
        unique_vals = set(df[col].unique())

        # Si los valores son subconjunto de Yes No, No phone service y No internet service
        if unique_vals.issubset({'Yes', 'No'}):
            cols_yes_no.append(col)

print(f"Columnas detectadas para convertir: {cols_yes_no}")

# 3. Aplicamos el reemplazo
for col in cols_yes_no:
    df[col] = df[col].map(mapa_booleano)


df.head()

Columnas detectadas para convertir: ['Churn', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']


,id,Churn,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,...,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total
0,0002-ORFBO,0,Female,0,1,1,9,1,No,DSL,...,Yes,No,Yes,Yes,No,One year,1,Mailed check,65.6,593.30
1,0003-MKNFE,0,Male,0,0,0,9,1,Yes,DSL,...,No,No,No,No,Yes,Month-to-month,0,Mailed check,59.9,542.40
2,0004-TLHLJ,1,Male,0,0,0,4,1,No,Fiber optic,...,No,Yes,No,No,No,Month-to-month,1,Electronic check,73.9,280.85
3,0011-IGKFF,1,Male,1,1,0,13,1,No,Fiber optic,...,Yes,Yes,No,Yes,Yes,Month-to-month,1,Electronic check,98.0,1237.85
4,0013-EXCHZ,1,Female,1,1,0,3,1,No,Fiber optic,...,No,No,Yes,Yes,No,Month-to-month,1,Mailed check,83.9,267.40


- Continuamos con las columnas restantes

In [74]:
# 1. Definimos el mapeo con un diccionario para aplicar los cambios
mapa = {'Yes': 1, 'No': 0,
        'No phone service': 0, 'No internet service': 0,
        }

# 2. Identificamos las columnas a normalizar
cols_normalizar = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                   'DeviceProtection', 'TechSupport', 'StreamingTV',
                   'StreamingMovies']




# 3. Aplicamos el reemplazo
for col in cols_normalizar:
    df[col] = df[col].map(mapa)


print(f"Columnas detectadas para convertir: {len(cols_normalizar)}")


df.head()

Columnas detectadas para convertir: 7


,id,Churn,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,...,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total
0,0002-ORFBO,0,Female,0,1,1,9,1,0,DSL,...,1,0,1,1,0,One year,1,Mailed check,65.6,593.30
1,0003-MKNFE,0,Male,0,0,0,9,1,1,DSL,...,0,0,0,0,1,Month-to-month,0,Mailed check,59.9,542.40
2,0004-TLHLJ,1,Male,0,0,0,4,1,0,Fiber optic,...,0,1,0,0,0,Month-to-month,1,Electronic check,73.9,280.85
3,0011-IGKFF,1,Male,1,1,0,13,1,0,Fiber optic,...,1,1,0,1,1,Month-to-month,1,Electronic check,98.0,1237.85
4,0013-EXCHZ,1,Female,1,1,0,3,1,0,Fiber optic,...,0,0,1,1,0,Month-to-month,1,Mailed check,83.9,267.40


In [75]:
# Verificación de valores unicos de cada columna categorica

for col in columnas_categoricas:
    print(f"Valores únicos en la columna {col}: {df[col].unique()}.")

Valores únicos en la columna Churn: [0 1].
Valores únicos en la columna Gender: ['Female' 'Male'].
Valores únicos en la columna SeniorCitizen: [0 1].
Valores únicos en la columna Partner: [1 0].
Valores únicos en la columna Dependents: [1 0].
Valores únicos en la columna PhoneService: [1 0].
Valores únicos en la columna MultipleLines: [0 1].
Valores únicos en la columna InternetService: ['DSL' 'Fiber optic' 'No'].
Valores únicos en la columna OnlineSecurity: [0 1].
Valores únicos en la columna OnlineBackup: [1 0].
Valores únicos en la columna DeviceProtection: [0 1].
Valores únicos en la columna TechSupport: [1 0].
Valores únicos en la columna StreamingTV: [1 0].
Valores únicos en la columna StreamingMovies: [0 1].
Valores únicos en la columna Contract: ['One year' 'Month-to-month' 'Two year'].
Valores únicos en la columna PaperlessBilling: [1 0].
Valores únicos en la columna PaymentMethod: ['Mailed check' 'Electronic check' 'Credit card (automatic)'
 'Bank transfer (automatic)'].


In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                7032 non-null   object 
 1   Churn             7032 non-null   int64  
 2   Gender            7032 non-null   object 
 3   SeniorCitizen     7032 non-null   int64  
 4   Partner           7032 non-null   int64  
 5   Dependents        7032 non-null   int64  
 6   Tenure            7032 non-null   int64  
 7   PhoneService      7032 non-null   int64  
 8   MultipleLines     7032 non-null   int64  
 9   InternetService   7032 non-null   object 
 10  OnlineSecurity    7032 non-null   int64  
 11  OnlineBackup      7032 non-null   int64  
 12  DeviceProtection  7032 non-null   int64  
 13  TechSupport       7032 non-null   int64  
 14  StreamingTV       7032 non-null   int64  
 15  StreamingMovies   7032 non-null   int64  
 16  Contract          7032 non-null   object 
 17  

### Crear columna 'cuentas_diarias'
Vamos a crear una columna que contenga los gastos diarios.

In [77]:
df['Charges.Daily'] = df['Charges.Monthly'] / 30
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7266
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                7032 non-null   object 
 1   Churn             7032 non-null   int64  
 2   Gender            7032 non-null   object 
 3   SeniorCitizen     7032 non-null   int64  
 4   Partner           7032 non-null   int64  
 5   Dependents        7032 non-null   int64  
 6   Tenure            7032 non-null   int64  
 7   PhoneService      7032 non-null   int64  
 8   MultipleLines     7032 non-null   int64  
 9   InternetService   7032 non-null   object 
 10  OnlineSecurity    7032 non-null   int64  
 11  OnlineBackup      7032 non-null   int64  
 12  DeviceProtection  7032 non-null   int64  
 13  TechSupport       7032 non-null   int64  
 14  StreamingTV       7032 non-null   int64  
 15  StreamingMovies   7032 non-null   int64  
 16  Contract          7032 non-null   object 
 17  

### 📊 Preprocesamiento

1) Análisis Descriptivo